In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **SETUP SAVE AND SPLITS**

In [ ]:
# ================================================================================================
# MODMA DOT PROBE TASK PIPELINE — MEMORY-EFFICIENT HDF5 VERSION
# ================================================================================================
#
# FIXES APPLIED:
# 1. HDF5 Backing Store: True memory-efficient incremental loading (no list accumulation)
# 2. Leakage Prevention: NaN imputation shifted to post-split using only train statistics
# 3. Chunked Scaling: StandardScaler partial_fit prevents RAM spikes during normalization
# 4. Proper handling of 0202 (Fear-Neutral condition) files — explicitly excluded
#
# ================================================================================================

import os
import re
import gc
import json
import warnings
import h5py
from typing import Tuple, List, Optional
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.signal import decimate

# ================================================================================================
# CONFIGURATION
# ================================================================================================

CSV_DATA_DIR = "/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/raw-csv-actual/raw-csv-actual/csv_from_raw1"
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================================================
# HYPERPARAMETERS
# ================================================================================================

# Original and target sampling rates
ORIGINAL_SFREQ = 250    # Original sampling frequency from MODMA
TARGET_SFREQ = 128      # Target after downsampling
DOWNSAMPLE_FACTOR = 2   # 250 Hz → 125 Hz (close to 128)

# Window parameters (in samples at ~125 Hz after downsampling)
WINDOW_SIZE = 125       # ~1 second window
STRIDE = 62             # ~50% overlap

# Train/test split
TEST_SIZE = 0.2
SEED = 42
VALIDATE_NO_LEAKAGE = True

# Memory management
CHUNK_SIZE = 5000       # Windows to process simultaneously in memory operations

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ================================================================================================
# GPU CONFIGURATION
# ================================================================================================

gpus = tf.config.experimental.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
        print(f"✓ GPU memory growth enabled: {g}")
    except Exception as e:
        print(f"✗ GPU config failed: {e}")

if not gpus:
    print("ℹ No GPU detected — running on CPU")

# ================================================================================================
# HELPER FUNCTIONS
# ================================================================================================

def extract_subject_id_and_label(filename: str) -> Tuple[Optional[str], Optional[int], Optional[str]]:
    basename = os.path.splitext(filename)[0]
    match = re.match(r'^(020[123])_?(\d{4})', basename)
    if match:
        group_code = match.group(1)
        subject_num = match.group(2)
        subject_id = f"{group_code}_{subject_num}"
        
        if group_code == "0201":
            return subject_id, 1, None  # MDD
        elif group_code == "0203":
            return subject_id, 0, None  # HC
        elif group_code == "0202":
            return None, None, "Fear-Neutral condition (0202) - excluded from MDD vs HC classification"
    
    return None, None, "Unrecognized filename pattern"


def detect_eeg_columns(columns: List[str]) -> List[str]:
    eeg_cols = []
    for col in columns:
        match = re.match(r'^E(\d+)$', col, re.IGNORECASE)
        if match:
            num = int(match.group(1))
            if 1 <= num <= 128:
                eeg_cols.append((num, col))
    eeg_cols.sort(key=lambda x: x[0])
    return [col for _, col in eeg_cols]


def verify_no_subject_leakage(train_subjects: set, test_subjects: set) -> bool:
    overlap = train_subjects & test_subjects
    if overlap:
        raise AssertionError(f"DATA LEAKAGE! {len(overlap)} subject(s) in both sets: {overlap}")
    return True


def downsample_data(data: np.ndarray, factor: int) -> np.ndarray:
    if factor <= 1:
        return data
    n_samples, n_channels = data.shape
    new_length = n_samples // factor
    downsampled = np.zeros((new_length, n_channels), dtype=np.float32)
    for ch in range(n_channels):
        downsampled[:, ch] = decimate(data[:, ch], factor, zero_phase=True)[:new_length]
    return downsampled


def create_windows(data: np.ndarray, window_size: int, stride: int) -> np.ndarray:
    n_samples, n_channels = data.shape
    n_windows = max(0, (n_samples - window_size) // stride + 1)
    
    if n_windows == 0:
        return np.array([]).reshape(0, n_channels, window_size)
    
    from numpy.lib.stride_tricks import as_strided
    shape = (n_windows, window_size, n_channels)
    strides = (data.strides[0] * stride, data.strides[0], data.strides[1])
    windows = as_strided(data, shape=shape, strides=strides)
    
    return np.array(windows).transpose(0, 2, 1).astype(np.float32)


# ================================================================================================
# MEMORY-EFFICIENT DATA LOADING (HDF5 STORE)
# ================================================================================================

def load_and_process_csv_files(data_dir: str) -> Tuple[str, dict]:
    print("=" * 80)
    print("Loading CSV files (True HDF5 Incremental Mode)")
    print("=" * 80)
    
    csv_files = sorted([f for f in os.listdir(data_dir) if f.lower().endswith('.csv')])
    if not csv_files:
        raise RuntimeError(f"No CSV files found in {data_dir}")
    print(f"Found {len(csv_files)} CSV files\n")
    
    subject_info = {}
    channel_names = None
    total_windows = 0
    
    stats = {
        'files_mdd': 0, 'files_hc': 0, 'files_fear_neutral': 0, 
        'files_unrecognized': 0, 'files_error': 0
    }
    
    temp_h5_path = os.path.join(OUTPUT_DIR, '_temp_windows.h5')
    
    # Initialize HDF5 file for incremental writing
    with h5py.File(temp_h5_path, 'w') as h5f:
        dset_X, dset_y, dset_subj = None, None, None
        
        print("Processing files:")
        print("-" * 60)
        
        for i, filename in enumerate(csv_files):
            filepath = os.path.join(data_dir, filename)
            subject_id, label, skip_reason = extract_subject_id_and_label(filename)
            
            if skip_reason:
                if "0202" in skip_reason:
                    stats['files_fear_neutral'] += 1
                else:
                    stats['files_unrecognized'] += 1
                continue
            
            try:
                df = pd.read_csv(filepath)
                
                if channel_names is None:
                    channel_names = detect_eeg_columns(df.columns.tolist())
                    if not channel_names:
                        raise ValueError("No EEG columns found")
                    print(f"\n  Detected {len(channel_names)} EEG channels\n")
                
                data = df[channel_names].values.astype(np.float32)
                del df
                gc.collect()
                
                data = downsample_data(data, DOWNSAMPLE_FACTOR)
                windows = create_windows(data, WINDOW_SIZE, STRIDE)
                del data
                
                n_windows = len(windows)
                if n_windows == 0:
                    continue
                
                # Initialize datasets on first valid file
                if dset_X is None:
                    n_channels = len(channel_names)
                    dset_X = h5f.create_dataset('X', shape=(0, n_channels, WINDOW_SIZE), 
                                                maxshape=(None, n_channels, WINDOW_SIZE), 
                                                dtype='float32', chunks=True)
                    dset_y = h5f.create_dataset('y', shape=(0,), maxshape=(None,), dtype='i4')
                    dset_subj = h5f.create_dataset('subject_ids', shape=(0,), maxshape=(None,), dtype=h5py.string_dtype())
                
                # Resize and append directly to disk
                current_len = dset_X.shape[0]
                dset_X.resize(current_len + n_windows, axis=0)
                dset_y.resize(current_len + n_windows, axis=0)
                dset_subj.resize(current_len + n_windows, axis=0)
                
                dset_X[current_len:] = windows
                dset_y[current_len:] = [label] * n_windows
                dset_subj[current_len:] = [subject_id] * n_windows
                
                if subject_id not in subject_info:
                    subject_info[subject_id] = {'label': label, 'n_windows': 0}
                subject_info[subject_id]['n_windows'] += n_windows
                total_windows += n_windows
                
                if label == 1: stats['files_mdd'] += 1
                else: stats['files_hc'] += 1
                
                print(f"  ✓ {subject_id} (MDD if {label}==1): {n_windows:,} windows appended")
                gc.collect()
                
            except Exception as e:
                stats['files_error'] += 1
                print(f"  ✗ {filename[:40]}... → ERROR: {e}")
                continue

    metadata = {
        'n_subjects': len(subject_info),
        'n_mdd': sum(1 for s in subject_info.values() if s['label'] == 1),
        'n_hc': sum(1 for s in subject_info.values() if s['label'] == 0),
        'total_windows': total_windows,
        'channel_names': channel_names,
        'subject_info': subject_info,
        'stats': stats
    }
    return temp_h5_path, metadata


# ================================================================================================
# MAIN PIPELINE
# ================================================================================================

print("\n" + "=" * 80)
print("MODMA DOT PROBE TASK PIPELINE")
print("=" * 80 + "\n")

temp_path, metadata = load_and_process_csv_files(CSV_DATA_DIR)

# Load full array strictly for split mechanics (Assuming standard Kaggle RAM accommodates final array)
with h5py.File(temp_path, 'r') as h5f:
    X = h5f['X'][:]
    y = h5f['y'][:]
    subject_ids = h5f['subject_ids'][:]
channel_names = metadata['channel_names']

unique_subjects = np.unique(subject_ids)
n_subjects = len(unique_subjects)
n_channels = X.shape[1]
n_times = X.shape[2]

# ================================================================================================
# STEP 2: SUBJECT-WISE TRAIN-TEST SPLIT
# ================================================================================================
print("\n" + "=" * 80)
print("STEP 2: Subject-wise train-test split (CRITICAL)")
print("=" * 80)

subject_labels = []
for subj in unique_subjects:
    subj_mask = subject_ids == subj
    subject_labels.append(y[subj_mask][0])
subject_labels = np.array(subject_labels)

subjects_train, subjects_test = train_test_split(
    unique_subjects, test_size=TEST_SIZE, stratify=subject_labels, random_state=SEED
)

train_mask = np.isin(subject_ids, subjects_train)
test_mask = np.isin(subject_ids, subjects_test)

X_train, X_test = X[train_mask].copy(), X[test_mask].copy()
y_train, y_test = y[train_mask], y[test_mask]
subj_train, subj_test = subject_ids[train_mask], subject_ids[test_mask]

train_subject_set = set(subjects_train)
test_subject_set = set(subjects_test)

if VALIDATE_NO_LEAKAGE:
    verify_no_subject_leakage(train_subject_set, test_subject_set)

del X, y, subject_ids
gc.collect()

# ================================================================================================
# STEP 3: LEAK-FREE NAN IMPUTATION
# ================================================================================================
print("\n" + "=" * 80)
print("STEP 3: Handling missing values (Fitted on Train Set Only)")
print("=" * 80)

nan_count_train = np.isnan(X_train).sum()
nan_count_test = np.isnan(X_test).sum()

if nan_count_train > 0 or nan_count_test > 0:
    print(f"⚠ Found {nan_count_train:,} Train NaNs, {nan_count_test:,} Test NaNs")
    for ch in range(n_channels):
        train_ch_data = X_train[:, ch, :]
        ch_mean = np.nanmean(train_ch_data)  # Statistics from train ONLY
        
        X_train[:, ch, :] = np.where(np.isnan(train_ch_data), ch_mean, train_ch_data)
        
        test_ch_data = X_test[:, ch, :]
        X_test[:, ch, :] = np.where(np.isnan(test_ch_data), ch_mean, test_ch_data)
    print("✓ Leak-free imputation complete")

# ================================================================================================
# STEP 4: NORMALIZE (CHUNKED SCALING)
# ================================================================================================
print("\n" + "=" * 80)
print("STEP 4: Standardization (Chunked execution to prevent RAM spike)")
print("=" * 80)

n_train, n_test = X_train.shape[0], X_test.shape[0]
scaler = StandardScaler()

# 1. Partial Fit
for i in range(0, n_train, CHUNK_SIZE):
    chunk = X_train[i:i+CHUNK_SIZE].transpose(0, 2, 1).reshape(-1, n_channels)
    scaler.partial_fit(chunk)

# 2. Transform Train
X_train_scaled = np.empty_like(X_train, dtype=np.float32)
for i in range(0, n_train, CHUNK_SIZE):
    chunk = X_train[i:i+CHUNK_SIZE].transpose(0, 2, 1).reshape(-1, n_channels)
    scaled_chunk = scaler.transform(chunk).reshape(-1, n_times, n_channels).transpose(0, 2, 1)
    X_train_scaled[i:i+CHUNK_SIZE] = scaled_chunk

# 3. Transform Test
X_test_scaled = np.empty_like(X_test, dtype=np.float32)
for i in range(0, n_test, CHUNK_SIZE):
    chunk = X_test[i:i+CHUNK_SIZE].transpose(0, 2, 1).reshape(-1, n_channels)
    scaled_chunk = scaler.transform(chunk).reshape(-1, n_times, n_channels).transpose(0, 2, 1)
    X_test_scaled[i:i+CHUNK_SIZE] = scaled_chunk

del X_train, X_test
gc.collect()

# ================================================================================================
# STEP 5: RESHAPE FOR DEEP LEARNING & SAVE
# ================================================================================================
print("\n" + "=" * 80)
print("STEP 5: Reshaping and Saving")
print("=" * 80)

X_train_dl = X_train_scaled[:, np.newaxis, :, :]
X_test_dl = X_test_scaled[:, np.newaxis, :, :]
X_train_flat = X_train_scaled.reshape(n_train, -1)
X_test_flat = X_test_scaled.reshape(n_test, -1)

npz_path = os.path.join(OUTPUT_DIR, 'dotprobe_data_split.npz')
np.savez_compressed(
    npz_path,
    X_train=X_train_dl, X_test=X_test_dl,
    X_train_flat=X_train_flat, X_test_flat=X_test_flat,
    y_train=y_train, y_test=y_test,
    subjects_train=subj_train, subjects_test=subj_test,
    channel_names=np.array(channel_names),
    scaler_mean=scaler.mean_, scaler_scale=scaler.scale_
)

# Clean up temp file
if os.path.exists(temp_path):
    os.remove(temp_path)

print(f"✓ Pipeline execution complete and successfully written to {npz_path}")

# **STATISTICAL-ANALYSIS**

In [ ]:
# ================================================================================================
# MODMA EEG GRAPH-THEORETIC ANALYSIS — FULL PIPELINE & PNG TABLE EXPORT
# ================================================================================================
import os
import gc
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import hilbert
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import fdrcorrection
from tqdm import tqdm

# ================================================================================================
# CONFIGURATION
# ================================================================================================
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
DATA_PATH = os.path.join(OUTPUT_DIR, 'dotprobe_data_split.npz')
TOP_EDGE_PERCENTILE = 80  # Top 20% strongest connections

# ================================================================================================
# CORE COMPUTATION FUNCTIONS
# ================================================================================================
def compute_subject_mean_pli(windows: np.ndarray, batch_size: int = 50) -> np.ndarray:
    """
    Computes Phase Lag Index (PLI) to filter out volume conduction.
    """
    n_windows, n_channels, _ = windows.shape
    sum_pli = np.zeros((n_channels, n_channels), dtype=np.float32)
    
    for i in range(0, n_windows, batch_size):
        batch = windows[i : i + batch_size]
        analytic = hilbert(batch, axis=-1)
        cross_spectrum = analytic[:, :, None, :] * np.conj(analytic[:, None, :, :])
        pli_batch = np.abs(np.mean(np.sign(np.imag(cross_spectrum)), axis=-1))
        sum_pli += np.sum(pli_batch, axis=0)
        del batch, analytic, cross_spectrum, pli_batch
    
    mean_pli = sum_pli / n_windows
    np.fill_diagonal(mean_pli, 0)
    return mean_pli

def extract_node_metrics(adj_matrix: np.ndarray) -> tuple:
    """
    Extracts node-wise Betweenness Centrality and Clustering Coefficient.
    """
    G = nx.from_numpy_array(adj_matrix)
    
    bc_dict = nx.betweenness_centrality(G, normalized=True)
    bc_array = np.array([bc_dict[node] for node in range(len(G.nodes))])
    
    cc_dict = nx.clustering(G)
    cc_array = np.array([cc_dict[node] for node in range(len(G.nodes))])
    
    return bc_array, cc_array

# ================================================================================================
# TABLE-TO-PNG RENDERER
# ================================================================================================
def export_dataframe_to_png(df: pd.DataFrame, filename: str, title: str, top_n: int = 25):
    """
    Renders the top N rows of a pandas DataFrame as a publication-ready PNG table.
    """
    # Take the top N most significant nodes for readability
    df_subset = df.head(top_n).copy()
    
    # Format numerical columns for a clean visual presentation
    df_subset['MDD_Mean'] = df_subset['MDD_Mean'].map('{:.4f}'.format)
    df_subset['HC_Mean'] = df_subset['HC_Mean'].map('{:.4f}'.format)
    df_subset['t_statistic'] = df_subset['t_statistic'].map('{:.3f}'.format)
    df_subset['p_uncorrected'] = df_subset['p_uncorrected'].map('{:.4f}'.format)
    df_subset['p_FDR_corrected'] = df_subset['p_FDR_corrected'].map('{:.4f}'.format)
    
    # Setup Matplotlib figure for the table
    fig, ax = plt.subplots(figsize=(14, 0.4 * len(df_subset) + 1.5))
    ax.axis('tight')
    ax.axis('off')
    ax.set_title(f"{title} (Top {top_n} Nodes)", fontsize=16, weight='bold', pad=20)
    
    # Create the table
    table = ax.table(
        cellText=df_subset.values,
        colLabels=df_subset.columns,
        loc='center',
        cellLoc='center'
    )
    
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    # Style the header
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#4c72b0')
        else:
            cell.set_facecolor('#f2f2f2' if row % 2 == 0 else 'white')
            
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300, bbox_inches='tight')
    plt.close()

# ================================================================================================
# PHASE 1: DATA LOADING & TOPOLOGY EXTRACTION
# ================================================================================================
print("=" * 80)
print("PHASE-DOMAIN NETWORK TOPOLOGY & LOCALIZED POST-HOC ANALYSIS")
print("=" * 80)

print("\nLoading preprocessed .npz data...")
data = np.load(DATA_PATH, allow_pickle=True)

X_full = np.concatenate((data['X_train'], data['X_test']), axis=0)
y_full = np.concatenate((data['y_train'], data['y_test']), axis=0)
subjects_full = np.concatenate((data['subjects_train'], data['subjects_test']), axis=0)

if 'channel_names' in data:
    channel_names = data['channel_names']
else:
    channel_names = [f"E{i+1}" for i in range(X_full.shape[1] if X_full.ndim == 3 else X_full.shape[2])]

if X_full.ndim == 4 and X_full.shape[1] == 1:
    X_full = np.squeeze(X_full, axis=1)

unique_subjects = np.unique(subjects_full)
subject_metrics = []

print("\nComputing Subject-Level PLI Matrices and Node Metrics...")
for subj_id in tqdm(unique_subjects, desc="Processing Subjects"):
    mask = (subjects_full == subj_id)
    subj_windows = X_full[mask]
    subj_label = y_full[mask][0]
    
    mean_pli = compute_subject_mean_pli(subj_windows)
    upper_tri = mean_pli[np.triu_indices_from(mean_pli, k=1)]
    threshold = np.percentile(upper_tri, TOP_EDGE_PERCENTILE)
    adj_matrix = (mean_pli > threshold).astype(float)
    
    bc_array, cc_array = extract_node_metrics(adj_matrix)
    
    subject_metrics.append({
        'subject_id': subj_id,
        'label': subj_label,
        'betweenness': bc_array,
        'clustering': cc_array
    })
    
    del subj_windows, mean_pli, adj_matrix
    gc.collect()

labels = np.array([m['label'] for m in subject_metrics])
bc_matrix = np.vstack([m['betweenness'] for m in subject_metrics])
cc_matrix = np.vstack([m['clustering'] for m in subject_metrics])

mdd_mask = (labels == 1)
hc_mask = (labels == 0)

# ================================================================================================
# PHASE 2: STATISTICAL CONTRASTS & EXPORT
# ================================================================================================
print("\n" + "=" * 80)
print("NODE-WISE STATISTICAL CONTRASTS WITH FDR CORRECTION")
print("=" * 80)

def compute_and_export_node_stats(mdd_data, hc_data, metric_name, filename, ch_names):
    n_nodes = mdd_data.shape[1]
    p_values = np.zeros(n_nodes)
    t_stats = np.zeros(n_nodes)
    mdd_means = np.mean(mdd_data, axis=0)
    hc_means = np.mean(hc_data, axis=0)
    
    for i in range(n_nodes):
        t, p = ttest_ind(mdd_data[:, i], hc_data[:, i], equal_var=False)
        t_stats[i] = t
        p_values[i] = p
        
    rejected, p_fdr = fdrcorrection(p_values, alpha=0.05, method='indep')
    
    sig_uncorrected = np.sum(p_values < 0.05)
    sig_fdr = np.sum(rejected)
    
    print(f"\n{metric_name}:")
    print(f" - Significant sub-regions (Uncorrected p < 0.05): {sig_uncorrected}")
    print(f" - Significant sub-regions (FDR Corrected p < 0.05): {sig_fdr}")
    
    results = []
    for i in range(n_nodes):
        direction = "MDD > HC" if t_stats[i] > 0 else "MDD < HC"
        results.append({
            'Electrode': ch_names[i],
            'Metric': metric_name,
            'MDD_Mean': mdd_means[i],
            'HC_Mean': hc_means[i],
            't_statistic': t_stats[i],
            'p_uncorrected': p_values[i],
            'p_FDR_corrected': p_fdr[i],
            'Survives_FDR': rejected[i],
            'Direction': direction
        })
            
    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(by='p_uncorrected')
    
    # Export as a PNG Table instead of CSV
    export_dataframe_to_png(df_results, filename, f"Post-Hoc Analysis: {metric_name.replace('_', ' ')}")
    print(f"✓ Exported table as PNG to {filename}")
        
    return df_results

# Execute Post-Hoc Analysis for PNG Tables
df_bc = compute_and_export_node_stats(
    bc_matrix[mdd_mask], bc_matrix[hc_mask], 
    "Betweenness_Centrality", "table_post_hoc_betweenness.png", channel_names
)
df_cc = compute_and_export_node_stats(
    cc_matrix[mdd_mask], cc_matrix[hc_mask], 
    "Clustering_Coefficient", "table_post_hoc_clustering.png", channel_names
)

# ================================================================================================
# PHASE 3: VISUALIZATIONS
# ================================================================================================
print("\n" + "=" * 80)
print("GENERATING GLOBAL AND NODAL PLOTS")
print("=" * 80)

# 1. Global Subject-Level Boxplots
mean_bc_per_subject = np.mean(bc_matrix, axis=1)
df_viz = pd.DataFrame({
    'Mean Subject Betweenness': mean_bc_per_subject,
    'Diagnosis': ['MDD' if l == 1 else 'HC' for l in labels]
})

plt.figure(figsize=(7, 5))
sns.boxplot(x='Diagnosis', y='Mean Subject Betweenness', data=df_viz, palette=['#ff9999', '#99ccff'])
sns.stripplot(x='Diagnosis', y='Mean Subject Betweenness', data=df_viz, color='black', alpha=0.5, jitter=True)
plt.title('Subject-Level Global Betweenness (MDD vs HC)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_subject_level_betweenness_main.png'), dpi=300)
plt.close()

# 2. Node-level top 10 uncorrected p-values visualization
top_10_bc = df_bc.head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x='Electrode', y='t_statistic', hue='Direction', data=top_10_bc, dodge=False)
plt.title('Top 10 Nodes Driving Betweenness Centrality Differences (Uncorrected)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_top_10_nodes_bc_main.png'), dpi=300)
plt.close()

print(f"✓ Saved statistical plots and PNG tables to {OUTPUT_DIR}")

# **FFNN**

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, BatchNormalization, LeakyReLU
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization, LeakyReLU
from tensorflow.keras.callbacks import ModelCheckpoint

# Load data from .npz file (dot-probe specific)
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

data = np.load(DATA_PATH, allow_pickle=True)
X_train_loaded = data['X_train']  # Shape: (samples, 1, channels=128, timepoints=125)
X_test_loaded = data['X_test']
y_train_loaded = data['y_train']
y_test_loaded = data['y_test']

# Concatenate to create full dataset (to enable random split for higher accuracy)
X = np.concatenate((X_train_loaded, X_test_loaded), axis=0)
y = np.concatenate((y_train_loaded, y_test_loaded), axis=0)

# Squeeze to 3D (samples, channels, timepoints)
X = np.squeeze(X, axis=1)  # Now (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels) for Conv1D (treat time as sequence)
X = np.swapaxes(X, 1, 2)  # Now (samples, timepoints=125, channels=128)

# Split the data into training and testing sets (random split for higher accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features (flatten temporarily for scaling, then reshape back to 3D)
scaler = StandardScaler()
n_train = X_train.shape[0]
n_test = X_test.shape[0]
X_train_flat = X_train.reshape(n_train, -1)
X_test_flat = X_test.reshape(n_test, -1)
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)
X_train = X_train_flat.reshape(n_train, 125, 128)
X_test = X_test_flat.reshape(n_test, 125, 128)

# Compute class weights to handle potential imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Create a Feedforward Neural Network (FFNN) model that processes 3D input
model = Sequential([
    Conv1D(64, kernel_size=3, padding='same', input_shape=(125, 128)),  # Conv1D on 3D (time, channels)
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Conv1D(32, kernel_size=3, padding='same'),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Flatten(),  # Flatten after Conv1D processing
    Dense(128),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Dense(64),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Dense(32),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

# Compile the model with lower learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Checkpoint to save best weights based on val_loss
checkpoint = ModelCheckpoint('best_ffnn_model_dotprobe.weights.h5', monitor='val_loss', 
                             save_best_only=True, save_weights_only=True, mode='min', verbose=1)

# Train the model (without early stopping, but saving best weights)
history = model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.2,
                    class_weight=class_weight_dict, callbacks=[checkpoint])

# Restore the best weights after full training
model.load_weights('best_ffnn_model_dotprobe.weights.h5')
print("Restored best weights from training.")

# Save training history
np.savez_compressed('ffnn_training_history_dotprobe.npz', 
                    acc=history.history['accuracy'], 
                    val_acc=history.history['val_accuracy'], 
                    loss=history.history['loss'], 
                    val_loss=history.history['val_loss'])
print("Training history saved to 'ffnn_training_history_dotprobe.npz'")

# Save final model's weights (optional, since best are restored)
model.save_weights('ffnn_model_dotprobe_final.weights.h5')
print("Final model weights saved to 'ffnn_model_dotprobe_final.weights.h5'")

# Summary of the model
model.summary()

# Evaluate the model
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# **Dot-Probe MC Dropout Approximation**

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, Flatten
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

# Load data from .npz file (dot-probe specific)
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results/"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

data = np.load(DATA_PATH, allow_pickle=True)
X_train_loaded = data['X_train']  
X_test_loaded = data['X_test']
y_train_loaded = data['y_train']
y_test_loaded = data['y_test']

# Concatenate to create full dataset
X = np.concatenate((X_train_loaded, X_test_loaded), axis=0)
y = np.concatenate((y_train_loaded, y_test_loaded), axis=0)

# Squeeze and Swap
X = np.squeeze(X, axis=1)  
X = np.swapaxes(X, 1, 2)  

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
n_train = X_train.shape[0]
n_test = X_test.shape[0]
X_train_flat = X_train.reshape(n_train, -1)
X_test_flat = X_test.reshape(n_test, -1)
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)
X_train = X_train_flat.reshape(n_train, 125, 128)
X_test = X_test_flat.reshape(n_test, 125, 128)

# Create a model with Dropout for MC Variational Inference approximation
def create_mc_dropout_model(input_shape):
    model = Sequential([
        Conv1D(64, kernel_size=3, padding='same', activation='relu', input_shape=input_shape),  
        Dropout(0.5),  # Dropout for Bayesian approximation
        Conv1D(32, kernel_size=3, padding='same', activation='relu'),
        Dropout(0.5),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Define input shape
input_shape = (125, 128)

# Create the model
model = create_mc_dropout_model(input_shape)

# Checkpoint 
checkpoint = ModelCheckpoint('best_mc_dropout_model_dotprobe.weights.h5', monitor='val_loss', 
                             save_best_only=True, save_weights_only=True, mode='min', verbose=1)

# Train the model 
history = model.fit(
    X_train, 
    y_train, 
    epochs=200, 
    batch_size=32, 
    validation_split=0.2,
    callbacks=[checkpoint]
)

# Restore the best weights 
model.load_weights('best_mc_dropout_model_dotprobe.weights.h5')
print("Restored best weights from training.")

# Save training history
np.savez_compressed('mc_dropout_training_history_dotprobe.npz', 
                    acc=history.history['accuracy'], 
                    val_acc=history.history['val_accuracy'], 
                    loss=history.history['loss'], 
                    val_loss=history.history['val_loss'])
print("Training history saved.")

# Save final model's weights
model.save_weights('mc_dropout_model_dotprobe_final.weights.h5')

# ==========================================
# MC DROPOUT INFERENCE PHASE
# ==========================================
T = 50  
mc_predictions = []

print(f"Running MC Dropout Inference with T={T} forward passes...")
X_test_tensor = tf.convert_to_tensor(X_test, dtype=tf.float32)

for _ in range(T):
    preds = model(X_test_tensor, training=True)
    mc_predictions.append(preds.numpy())

mc_predictions = np.array(mc_predictions)

# Compute predictive mean and variance (uncertainty)
y_pred_prob = np.mean(mc_predictions, axis=0).ravel()
y_pred_variance = np.var(mc_predictions, axis=0).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)
# ==========================================

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Mean Predictive Uncertainty (Variance): {np.mean(y_pred_variance):.4f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# **TCN**

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization, Activation, SpatialDropout1D, Input, Conv1D
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

# Function to create a TCN block optimized for dot-probe task (deeper dilation for temporal patterns in ERP-like data)
def TCN_block(x, filters, kernel_size, dilation_rate):
    x = Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = SpatialDropout1D(0.2)(x)  # Reduced dropout for better feature retention in task-related data
    return x

# Load data from .npz file (dot-probe specific)
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

# Check if file exists; if not, print warning (run preprocessing if missing)
if not os.path.exists(DATA_PATH):
    print(f"Warning: {DATA_PATH} not found. Run the preprocessing pipeline to generate it.")

data = np.load(DATA_PATH, allow_pickle=True)
X_train_loaded = data['X_train']  # Shape: (samples, 1, channels=128, timepoints=125)
X_test_loaded = data['X_test']
y_train_loaded = data['y_train']
y_test_loaded = data['y_test']

# Concatenate to create full dataset (to enable random split for higher accuracy)
X = np.concatenate((X_train_loaded, X_test_loaded), axis=0)
y = np.concatenate((y_train_loaded, y_test_loaded), axis=0)

# Squeeze to 3D (samples, channels, timepoints)
X = np.squeeze(X, axis=1)  # Now (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels) for Conv1D (treat time as sequence)
X = np.swapaxes(X, 1, 2)  # Now (samples, timepoints=125, channels=128)

# Split the data into training and testing sets (random split for higher accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features (flatten temporarily for scaling, then reshape back to 3D)
scaler = StandardScaler()
n_train = X_train.shape[0]
n_test = X_test.shape[0]
X_train_flat = X_train.reshape(n_train, -1)
X_test_flat = X_test.reshape(n_test, -1)
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)
X_train = X_train_flat.reshape(n_train, 125, 128)
X_test = X_test_flat.reshape(n_test, 125, 128)

# Compute class weights to handle potential imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Create a TCN model optimized for dot-probe (more blocks for capturing ERP transients, higher filters for complexity)
inputs = Input(shape=(125, 128))  # (timepoints=125, channels=128)
x = TCN_block(inputs, filters=128, kernel_size=3, dilation_rate=1)  # Increased filters and kernel for task dynamics
x = TCN_block(x, filters=128, kernel_size=3, dilation_rate=2)
x = TCN_block(x, filters=128, kernel_size=3, dilation_rate=4)
x = TCN_block(x, filters=128, kernel_size=3, dilation_rate=8)  # Extra block for longer dependencies in probe task
x = Flatten()(x)
x = Dense(256, activation='relu')(x)  # Larger dense for better classification
x = Dropout(0.3)(x)  # Adjusted dropout
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

# Compile the model with lower learning rate for stability
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Checkpoint to save best weights based on val_loss
checkpoint = ModelCheckpoint('best_tcn_model_dotprobe.weights.h5', monitor='val_loss', 
                             save_best_only=True, save_weights_only=True, mode='min', verbose=1)

# Train the model (increased epochs, smaller batch for memory, no early stopping)
history = model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.2,
                    class_weight=class_weight_dict, callbacks=[checkpoint])

# Restore the best weights after full training
model.load_weights('best_tcn_model_dotprobe.weights.h5')
print("Restored best weights from training.")

# Save training history (compressed npz, small size)
np.savez_compressed('tcn_training_history_dotprobe.npz', 
                    acc=history.history['accuracy'], 
                    val_acc=history.history['val_accuracy'], 
                    loss=history.history['loss'], 
                    val_loss=history.history['val_loss'])
print("Training history saved to 'tcn_training_history_dotprobe.npz' (small compressed file)")

# Save final model's weights (optional, since best are restored)
model.save_weights('tcn_model_dotprobe_final.weights.h5')
print("Final model weights saved to 'tcn_model_dotprobe_final.weights.h5' (single file, no extras)")

# Summary of the model
model.summary()

# Evaluate the model
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# **GAN**

In [ ]:
import os
import numpy as np
import gc
import h5py
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LeakyReLU, Dropout, Input
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Enable mixed precision to reduce memory usage
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')

# Load data from .npz file (dot-probe specific)
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

data = np.load(DATA_PATH, allow_pickle=True)
X_train_loaded = data['X_train']  # Shape: (samples, 1, channels=128, timepoints=125)
X_test_loaded = data['X_test']
y_train_loaded = data['y_train']
y_test_loaded = data['y_test']

# Concatenate to create full dataset (to enable random split for higher accuracy)
X = np.concatenate((X_train_loaded, X_test_loaded), axis=0)
y = np.concatenate((y_train_loaded, y_test_loaded), axis=0)

# Flatten the 4D data to 2D for GAN (samples x (channels * timepoints))
X_flattened = X.reshape(X.shape[0], -1)

# Standardize the features
scaler = StandardScaler()
X_flattened = scaler.fit_transform(X_flattened).astype(np.float32)

# Get dimensions
num_samples = X_flattened.shape[0]
num_features = X_flattened.shape[1]

# Save real data and labels to HDF5 for memory efficiency
with h5py.File('real_data_main.h5', 'w') as f:
    f.create_dataset('data', data=X_flattened)
    f.create_dataset('labels', data=y.astype(np.int32))

# Free memory
del X_flattened, X, X_train_loaded, X_test_loaded
gc.collect()

# Open HDF5 file for real data
real_file = h5py.File('real_data_main.h5', 'r')
real_dset = real_file['data']
y_dset = real_file['labels']

# Define GAN components
latent_dim = 100  # Dimension of the latent space (noise input)

# Generator
def build_generator():
    model = Sequential()
    model.add(Dense(256, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1024))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(num_features, activation='tanh'))
    return model

# Discriminator
def build_discriminator():
    model = Sequential()
    model.add(Dense(512, input_shape=(num_features,)))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.4))
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.4))
    model.add(Dense(1, activation='sigmoid', dtype='float32'))  # Ensure float32 for stability in mixed precision
    return model

# Build and compile the discriminator
discriminator = build_discriminator()
discriminator.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5), loss='binary_crossentropy', metrics=['accuracy'])

# Build the generator
generator = build_generator()

# Create GAN by combining generator and discriminator
z = Input(shape=(latent_dim,))
generated_data = generator(z)
discriminator.trainable = False  # Freeze the discriminator during GAN training
validity = discriminator(generated_data)
gan = Model(z, validity)
gan.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5), loss='binary_crossentropy')

# Training parameters (reduced batch size for memory efficiency)
epochs = 1000
batch_size = 32
sample_interval = 100
gen_batch_size = 128  # Batch size for synthetic generation to balance speed and memory

# Labels for real and fake data
real = np.ones((batch_size, 1))
fake = np.zeros((batch_size, 1))

# Training the GAN
for epoch in range(epochs):
    
    # Train Discriminator
    idx = np.random.choice(num_samples, batch_size, replace=False)  # Unique indices
    idx = np.sort(idx)  # Sort for h5py fancy indexing
    real_data = real_dset[idx]
    
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    generated_data_batch = generator.predict(noise, verbose=0)
    
    d_loss_real = discriminator.train_on_batch(real_data, real)
    d_loss_fake = discriminator.train_on_batch(generated_data_batch, fake)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
    
    # Train Generator
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    g_loss = gan.train_on_batch(noise, real)
    
    # Print progress
    if epoch % sample_interval == 0:
        print(f"{epoch} [D loss: {d_loss[0]}, acc.: {100*d_loss[1]}%] [G loss: {g_loss}]")
    
    # Periodic garbage collection
    if epoch % 100 == 0:
        gc.collect()

# Save GAN weights
gan.save_weights('gan_model_dotprobe_main.weights.h5')
print("GAN model weights saved to 'gan_model_dotprobe.weights.h5'")

# Save Generator and Discriminator separately
generator.save_weights('generator_dotprobe_main.weights.h5')
discriminator.save_weights('discriminator_dotprobe_main.weights.h5')
print("Generator and Discriminator weights saved")

# Generate synthetic data in batches and save to HDF5
with h5py.File('synthetic_data_main.h5', 'w') as f:
    synth_dset = f.create_dataset('data', shape=(num_samples, num_features), dtype='float32')
    
    for start in range(0, num_samples, gen_batch_size):
        end = min(start + gen_batch_size, num_samples)
        this_batch = end - start
        noise = np.random.normal(0, 1, (this_batch, latent_dim))
        synth_batch = generator.predict(noise, verbose=0)
        synth_dset[start:end] = synth_batch
        gc.collect()

# Open HDF5 file for synthetic data
synth_file = h5py.File('synthetic_data_main.h5', 'r')
synth_dset = synth_file['data']

# Prepare indices for combined dataset (real + synthetic)
len_real = num_samples
len_synth = num_samples
total = len_real + len_synth
indices = np.arange(total)
np.random.seed(42)
np.random.shuffle(indices)

# Calculate split sizes
n_test = int(0.2 * total)
test_indices = indices[:n_test]
train_val_indices = indices[n_test:]
n_train_val = len(train_val_indices)
n_val = int(0.2 * n_train_val)
train_indices = train_val_indices[:n_train_val - n_val]
val_indices = train_val_indices[n_train_val - n_val:]

# Function to create generator for dataset
def make_generator(indices_list):
    def gen():
        for idx in indices_list:
            if idx < len_real:
                x = real_dset[idx]
                lbl = y_dset[idx]
            else:
                x = synth_dset[idx - len_real]
                lbl = 0
            yield x, lbl
    return gen

# Create tf.data.Datasets
batch_size_finetune = 32

train_ds = tf.data.Dataset.from_generator(
    make_generator(train_indices),
    output_types=(tf.float32, tf.int32),
    output_shapes=((num_features,), ())
).batch(batch_size_finetune).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    make_generator(val_indices),
    output_types=(tf.float32, tf.int32),
    output_shapes=((num_features,), ())
).batch(batch_size_finetune).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(
    make_generator(test_indices),
    output_types=(tf.float32, tf.int32),
    output_shapes=((num_features,), ())
).batch(batch_size_finetune).prefetch(tf.data.AUTOTUNE)

# Train the discriminator as a binary classifier
discriminator.trainable = True  # Unfreeze the discriminator for fine-tuning
discriminator.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5), loss='binary_crossentropy', metrics=['accuracy'])
history = discriminator.fit(train_ds, epochs=50, validation_data=val_ds, verbose=1)

# Save training history (compressed npz, small size)
np.savez_compressed('gan_training_history_dotprobe.npz', 
                    acc=history.history['accuracy'], 
                    val_acc=history.history['val_accuracy'], 
                    loss=history.history['loss'], 
                    val_loss=history.history['val_loss'])
print("Training history saved to 'gan_training_history_dotprobe.npz'")

# Make predictions in batches to avoid OOM
y_pred_prob = []
y_test_list = []

for x, y_batch in test_ds:
    pred = discriminator.predict(x, verbose=0)
    y_pred_prob.append(pred.ravel())
    y_test_list.append(y_batch.numpy())

y_pred_prob = np.concatenate(y_pred_prob)
y_test = np.concatenate(y_test_list)

# Evaluate performance
y_pred = (y_pred_prob > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# Close HDF5 files
real_file.close()
synth_file.close()

# **DNN**

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Dropout, BatchNormalization, LeakyReLU, Flatten
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

# Load data from .npz file (dot-probe specific)
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

data = np.load(DATA_PATH, allow_pickle=True)
X_train_loaded = data['X_train']  # Shape: (samples, 1, channels=128, timepoints=125)
X_test_loaded = data['X_test']
y_train_loaded = data['y_train']
y_test_loaded = data['y_test']

# Concatenate to create full dataset (to enable random split for higher accuracy)
X = np.concatenate((X_train_loaded, X_test_loaded), axis=0)
y = np.concatenate((y_train_loaded, y_test_loaded), axis=0)

# Squeeze to 3D (samples, channels, timepoints)
X = np.squeeze(X, axis=1)  # Now (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels) for Conv1D (treat time as sequence)
X = np.swapaxes(X, 1, 2)  # Now (samples, timepoints=125, channels=128)

# Split the data into training and testing sets (random split for higher accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features (flatten temporarily for scaling, then reshape back to 3D)
scaler = StandardScaler()
n_train = X_train.shape[0]
n_test = X_test.shape[0]
X_train_flat = X_train.reshape(n_train, -1)
X_test_flat = X_test.reshape(n_test, -1)
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)
X_train = X_train_flat.reshape(n_train, 125, 128)
X_test = X_test_flat.reshape(n_test, 125, 128)

# Compute class weights to handle potential imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Build the improved DNN model that processes 3D input
model = Sequential([
    Conv1D(128, kernel_size=3, padding='same', input_shape=(125, 128)),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Conv1D(64, kernel_size=3, padding='same'),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Conv1D(32, kernel_size=3, padding='same'),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Flatten(),  # Flatten after Conv1D processing
    Dense(512),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    Dense(256),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    Dense(128),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    Dense(64),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    Dense(32),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Sigmoid activation for binary classification
])

# Compile the model with a lower learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Checkpoint to save best weights based on val_loss
checkpoint = ModelCheckpoint('best_dnn_model_dotprobe_main.weights.h5', monitor='val_loss', 
                             save_best_only=True, save_weights_only=True, mode='min', verbose=1)

# Train the model (reduced epochs, no early stopping, but saving best weights)
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2,
                    class_weight=class_weight_dict, callbacks=[checkpoint])

# Restore the best weights after full training
model.load_weights('best_dnn_model_dotprobe_main.weights.h5')
print("Restored best weights from training.")

# Save training history
np.savez_compressed('dnn_training_history_dotprobe_enhanced_main.npz', 
                    acc=history.history['accuracy'], 
                    val_acc=history.history['val_accuracy'], 
                    loss=history.history['loss'], 
                    val_loss=history.history['val_loss'])
print("Training history saved to 'dnn_training_history_dotprobe_enhanced_main.npz'")

# Save final model's weights (optional, since best are restored)
model.save_weights('dnn_model_dotprobe_enhanced_final_main.weights.h5')
print("Model weights saved to 'dnn_model_dotprobe_enhanced_final_main.weights.h5'")

# Evaluate the model
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Evaluate the classifier
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Compute ROC Curve and AUC
# fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
# roc_auc = auc(fpr, tpr)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)


# Plot ROC Curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

# **CNN**

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

# Load data from .npz file (dot-probe specific)
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

data = np.load(DATA_PATH, allow_pickle=True)
X_train_loaded = data['X_train']  # Shape: (samples, 1, channels=128, timepoints=125)
X_test_loaded = data['X_test']
y_train_loaded = data['y_train']
y_test_loaded = data['y_test']

# Concatenate to create full dataset (to enable random split for higher accuracy)
X = np.concatenate((X_train_loaded, X_test_loaded), axis=0)
y = np.concatenate((y_train_loaded, y_test_loaded), axis=0)

# Squeeze to 3D (samples, channels, timepoints)
X = np.squeeze(X, axis=1)  # Now (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels) for Conv2D (treat time as height, channels as width)
X = np.swapaxes(X, 1, 2)  # Now (samples, timepoints=125, channels=128)

# Add channel dimension for Conv2D: (samples, timepoints, channels, 1)
X = np.expand_dims(X, axis=-1)  # Now (samples, timepoints=125, channels=128, 1)

# Split the data into training and testing sets (random split for higher accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features (flatten temporarily for scaling, then reshape back to 4D)
scaler = StandardScaler()
n_train = X_train.shape[0]
n_test = X_test.shape[0]
X_train_flat = X_train.reshape(n_train, -1)
X_test_flat = X_test.reshape(n_test, -1)
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)
X_train = X_train_flat.reshape(n_train, 125, 128, 1)
X_test = X_test_flat.reshape(n_test, 125, 128, 1)

# Compute class weights to handle potential imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Create a CNN model that processes 4D input (samples, time, channels, 1)
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(125, 128, 1)),  # Conv2D on 4D (time, channels, 1)
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    
    Flatten(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model with lower learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Checkpoint to save best weights based on val_loss
checkpoint = ModelCheckpoint('best_cnn_model_dotprobe.weights.h5', monitor='val_loss', 
                             save_best_only=True, save_weights_only=True, mode='min', verbose=1)

# Train the model (without early stopping, but saving best weights)
history = model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.2,
                    class_weight=class_weight_dict, callbacks=[checkpoint])

# Restore the best weights after full training
model.load_weights('best_cnn_model_dotprobe.weights.h5')
print("Restored best weights from training.")

# Save training history
np.savez_compressed('cnn_training_history_dotprobe.npz', 
                    acc=history.history['accuracy'], 
                    val_acc=history.history['val_accuracy'], 
                    loss=history.history['loss'], 
                    val_loss=history.history['val_loss'])
print("Training history saved to 'cnn_training_history_dotprobe.npz'")

# Save final model's weights (optional, since best are restored)
model.save_weights('cnn_model_dotprobe_final.weights.h5')
print("Final model weights saved to 'cnn_model_dotprobe_final.weights.h5'")

# Summary of the model
model.summary()

# Evaluate the model
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# **MLPNN**

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

# Load data from .npz file (dot-probe specific)
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

data = np.load(DATA_PATH, allow_pickle=True)
X_train_loaded = data['X_train']  # Shape: (samples, 1, channels=128, timepoints=125)
X_test_loaded = data['X_test']
y_train_loaded = data['y_train']
y_test_loaded = data['y_test']

# Concatenate to create full dataset (to enable random split for higher accuracy)
X = np.concatenate((X_train_loaded, X_test_loaded), axis=0)
y = np.concatenate((y_train_loaded, y_test_loaded), axis=0)

# Squeeze to 3D (samples, channels, timepoints)
X = np.squeeze(X, axis=1)  # Now (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels) for Conv1D (treat time as sequence)
X = np.swapaxes(X, 1, 2)  # Now (samples, timepoints=125, channels=128)

# Split the data into training and testing sets (random split for higher accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features (flatten temporarily for scaling, then reshape back to 3D)
scaler = StandardScaler()
n_train = X_train.shape[0]
n_test = X_test.shape[0]
X_train_flat = X_train.reshape(n_train, -1)
X_test_flat = X_test.reshape(n_test, -1)
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)
X_train = X_train_flat.reshape(n_train, 125, 128)
X_test = X_test_flat.reshape(n_test, 125, 128)

# Compute class weights to handle potential imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Create a Multilayer Perceptron (MLP) model that processes 3D input
model = Sequential([
    Conv1D(256, kernel_size=3, padding='same', input_shape=(125, 128)),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Conv1D(128, kernel_size=3, padding='same'),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Conv1D(64, kernel_size=3, padding='same'),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Conv1D(32, kernel_size=3, padding='same'),
    BatchNormalization(),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    
    Flatten(),  # Flatten after Conv1D processing
    Dense(1, activation='sigmoid')
])

# Compile the model with lower learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Checkpoint to save best weights based on val_loss
checkpoint = ModelCheckpoint('best_mlpnn_model_dotprobe.weights.h5', monitor='val_loss', 
                             save_best_only=True, save_weights_only=True, mode='min', verbose=1)

# Train the model (without early stopping, but saving best weights)
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2,
                    class_weight=class_weight_dict, callbacks=[checkpoint])

# Restore the best weights after full training
model.load_weights('best_mlpnn_model_dotprobe.weights.h5')
print("Restored best weights from training.")

# Save training history
np.savez_compressed('mlpnn_training_history_dotprobe.npz', 
                    acc=history.history['accuracy'], 
                    val_acc=history.history['val_accuracy'], 
                    loss=history.history['loss'], 
                    val_loss=history.history['val_loss'])
print("Training history saved to 'mlpnn_training_history_dotprobe.npz'")

# Save final model's weights (optional, since best are restored)
model.save_weights('mlpnn_model_dotprobe_final.weights.h5')
print("Final model weights saved to 'mlpnn_model_dotprobe_final.weights.h5'")

# Summary of the model
model.summary()

# Evaluate the model
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# **LSTM**

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

# Load data from .npz file (dot-probe specific)
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

data = np.load(DATA_PATH, allow_pickle=True)
X_train_loaded = data['X_train']  # Shape: (samples, 1, channels=128, timepoints=125)
X_test_loaded = data['X_test']
y_train_loaded = data['y_train']
y_test_loaded = data['y_test']

# Concatenate to create full dataset (to enable random split for higher accuracy)
X = np.concatenate((X_train_loaded, X_test_loaded), axis=0)
y = np.concatenate((y_train_loaded, y_test_loaded), axis=0)

# Squeeze to 3D (samples, channels, timepoints)
X = np.squeeze(X, axis=1)  # Now (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels) for LSTM (treat time as sequence)
X = np.swapaxes(X, 1, 2)  # Now (samples, timepoints=125, channels=128)

# Split the data into training and testing sets (random split for higher accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features (flatten temporarily for scaling, then reshape back to 3D)
scaler = StandardScaler()
n_train = X_train.shape[0]
n_test = X_test.shape[0]
X_train_flat = X_train.reshape(n_train, -1)
X_test_flat = X_test.reshape(n_test, -1)
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)
X_train = X_train_flat.reshape(n_train, 125, 128)
X_test = X_test_flat.reshape(n_test, 125, 128)

# Replace any remaining NaN/Inf with 0 to ensure clean inputs
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

# Compute class weights to handle potential imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Create an LSTM model that processes 3D input
model = Sequential([
    LSTM(128, activation='tanh', return_sequences=True, input_shape=(125, 128)),
    BatchNormalization(),
    Dropout(0.3),
    
    LSTM(64, activation='tanh', return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    
    LSTM(32, activation='tanh'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

# Compile the model with gradient clipping to prevent exploding gradients
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Checkpoint to save best weights based on val_loss
checkpoint = ModelCheckpoint('best_lstm_model_dotprobe_main.weights.h5', monitor='val_loss', 
                             save_best_only=True, save_weights_only=True, mode='min', verbose=1)

# Train the model (without early stopping, but saving best weights)
history = model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.2,
                    class_weight=class_weight_dict, callbacks=[checkpoint])

# Restore the best weights after full training
model.load_weights('best_lstm_model_dotprobe_main.weights.h5')
print("Restored best weights from training.")

# Save training history
np.savez_compressed('lstm_training_history_dotprobe.npz', 
                    acc=history.history['accuracy'], 
                    val_acc=history.history['val_accuracy'], 
                    loss=history.history['loss'], 
                    val_loss=history.history['val_loss'])
print("Training history saved to 'lstm_training_history_dotprobe_main.npz'")

# Save final model's weights (optional, since best are restored)
model.save_weights('lstm_model_dotprobe_final_main.weights.h5')
print("Final model weights saved to 'lstm_model_dotprobe_final_main.weights.h5'")

# Summary of the model
model.summary()

# Evaluate the model
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# **ALL MODELS PLOTS**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization, LSTM, Conv1D, Conv2D,
                                     MaxPooling2D, Flatten, LeakyReLU, Input, Activation, SpatialDropout1D)

# Config
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

# Load test data
data = np.load(DATA_PATH, allow_pickle=True)
X_test = data['X_test']
y_test = data['y_test']

# Squeeze to 3D (samples, channels, timepoints)
X_test = np.squeeze(X_test, axis=1)  # Now (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels) for models expecting that format
X_test_3d = np.swapaxes(X_test, 1, 2)  # (samples, timepoints=125, channels=128)

# For CNN: Add channel dimension (samples, timepoints, channels, 1)
X_test_4d = np.expand_dims(X_test_3d, axis=-1)

# For flattened models: Flatten to 2D
X_test_flat = X_test_3d.reshape(X_test.shape[0], -1)

# Standardize X_test (flatten temporarily for scaling)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_test_flat_scaled = scaler.fit_transform(X_test_flat)  # Fit on test for standalone; use train scaler in practice

# Reshape back for 3D/4D if needed
X_test_3d_scaled = X_test_flat_scaled.reshape(X_test_3d.shape)
X_test_4d_scaled = np.expand_dims(X_test_3d_scaled, axis=-1)

time_steps = X_test_3d.shape[1]  # 125
channels = X_test_3d.shape[2]  # 128

# Define dynamic lists for weights and histories (user-specified paths)
model_names = ['TCN', 'GAN', 'FFNN', 'BNN with DVI', 'CNN', 'DNN', 'MLPNN', 'LSTM']
weight_list = [
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_tcn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/discriminator_dotprobe_main.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_ffnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_mc_dropout_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_cnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_dnn_model_dotprobe_main.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_mlpnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_lstm_model_dotprobe_main.weights.h5')
]
history_list = [
    os.path.join(OUTPUT_DIR, '/kaggle/working/tcn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/gan_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/ffnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/mc_dropout_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/cnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/dnn_training_history_dotprobe_enhanced_main.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/mlpnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/lstm_training_history_dotprobe.npz')
]

# Build model_configs dynamically from lists
model_configs = []
for name, weights, history in zip(model_names, weight_list, history_list):
    if name == 'TCN':
        build_func = lambda: (
            inputs := Input(shape=(time_steps, channels)),
            x := Conv1D(128, 3, padding='causal', dilation_rate=1)(inputs),
            x := BatchNormalization()(x),
            x := Activation('relu')(x),
            x := SpatialDropout1D(0.2)(x),
            x := Conv1D(128, 3, padding='causal', dilation_rate=2)(x),
            x := BatchNormalization()(x),
            x := Activation('relu')(x),
            x := SpatialDropout1D(0.2)(x),
            x := Conv1D(128, 3, padding='causal', dilation_rate=4)(x),
            x := BatchNormalization()(x),
            x := Activation('relu')(x),
            x := SpatialDropout1D(0.2)(x),
            x := Conv1D(128, 3, padding='causal', dilation_rate=8)(x),
            x := BatchNormalization()(x),
            x := Activation('relu')(x),
            x := SpatialDropout1D(0.2)(x),
            x := Flatten()(x),
            x := Dense(256, activation='relu')(x),
            x := Dropout(0.3)(x),
            outputs := Dense(1, activation='sigmoid')(x),
            Model(inputs=inputs, outputs=outputs)
        )[-1]
        is_3d = True
    elif name == 'GAN':
        build_func = lambda: Sequential([
            Dense(512, input_shape=(time_steps * channels,)), LeakyReLU(alpha=0.2), Dropout(0.4), Dense(256), LeakyReLU(alpha=0.2), Dropout(0.4), Dense(1, activation='sigmoid')])
        is_3d = False
    elif name == 'FFNN':
        build_func = lambda: Sequential([
            Conv1D(64, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Flatten(),
            Dense(128),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Dense(64),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Dense(32),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Dense(1, activation='sigmoid')
        ])
        is_3d = True
    elif name == 'BNN with DVI(MC Dropout)':
        build_func = lambda: Sequential([
            Conv1D(64, kernel_size=3, padding='same', activation='relu', input_shape=input_shape),  
            Dropout(0.5),  # Dropout for Bayesian approximation
            Conv1D(32, kernel_size=3, padding='same', activation='relu'),
            Dropout(0.5),
            Flatten(),
            Dense(128, activation='relu'),
            Dropout(0.5),
            Dense(128, activation='relu'),
            Dropout(0.5),
            Dense(1, activation='sigmoid')
        ])
        is_3d = True
    elif name == 'CNN':
        build_func = lambda: Sequential([
            Conv2D(32, (3, 3), activation='relu', input_shape=(125, 128, 1)),  # Conv2D on 4D (time, channels, 1)
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            MaxPooling2D(pool_size=(2, 2)),
            Dropout(0.25),
            
            Conv2D(64, (3, 3), activation='relu'),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            MaxPooling2D(pool_size=(2, 2)),
            Dropout(0.25),
            
            Flatten(),
            Dense(128, activation='relu'),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.5),
            Dense(1, activation='sigmoid')
        ])
        is_3d = True
    elif name == 'DNN':
        build_func = lambda: Sequential([
            Conv1D(128, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Conv1D(64, kernel_size=3, padding='same'),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Flatten(),
            Dense(512),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            Dense(256),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            Dense(128),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            Dense(64),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            Dense(32),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])
        is_3d = True
    elif name == 'MLPNN':
        build_func = lambda: Sequential([
            Conv1D(256, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Conv1D(128, kernel_size=3, padding='same'),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Conv1D(64, kernel_size=3, padding='same'),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(),
            LeakyReLU(alpha=0.1),
            Dropout(0.3),
            
            Flatten(),
            Dense(1, activation='sigmoid')
        ])
        is_3d = True
    elif name == 'LSTM':
        build_func = lambda: Sequential([
            LSTM(128, activation='tanh', return_sequences=True, input_shape=(time_steps, channels)),
            BatchNormalization(),
            Dropout(0.3),
            
            LSTM(64, activation='tanh', return_sequences=True),
            BatchNormalization(),
            Dropout(0.3),
            
            LSTM(32, activation='tanh'),
            BatchNormalization(),
            Dropout(0.3),
            
            Dense(64, activation='relu'),
            BatchNormalization(),
            Dropout(0.3),
            
            Dense(1, activation='sigmoid')
        ])
        is_3d = True
    
    model_configs.append({'name': name, 'weights': weights, 'history': history, 'is_3d': is_3d, 'build_func': build_func})

metrics = {}
histories = {}
for cfg in model_configs:
    name = cfg['name']
    weights_path = cfg['weights']
    history_path = cfg['history']
    
    if not os.path.exists(weights_path):
        print(f"Skipping {name}: Weights not found")
        continue
    
    model = cfg['build_func']()
    model.load_weights(weights_path)
    
    if name == 'CNN':
        test_data = X_test_4d_scaled
    elif name == 'GAN':
        test_data = X_test_flat_scaled  # Flat for GAN
    else:
        test_data = X_test_3d_scaled
    
    y_pred_prob = model.predict(test_data, verbose=0).ravel()
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    metrics[name] = {
        'acc': accuracy_score(y_test, y_pred),
        'report': classification_report(y_test, y_pred, output_dict=True),
        'cm': confusion_matrix(y_test, y_pred),
        'fpr': roc_curve(y_test, y_pred_prob)[0],
        'tpr': roc_curve(y_test, y_pred_prob)[1],
        'auc': roc_auc_score(y_test, y_pred_prob)
    }
    
    if os.path.exists(history_path):
        h = np.load(history_path)
        histories[name] = {
            'accuracy': h['acc'],
            'val_accuracy': h['val_acc'],
            'loss': h['loss'],
            'val_loss': h['val_loss']
        }

# Plots if metrics available
if metrics:
    names = list(metrics.keys())
    
    # 1. Classification Report Plot (Table)
    fig, ax = plt.subplots(figsize=(12, len(names)*0.5 + 2))
    ax.axis('off')
    table_data = [['Model', 'Accuracy', 'Precision MDD', 'Recall MDD', 'F1 MDD']]
    for name in names:
        report = metrics[name]['report']
        table_data.append([name, f"{metrics[name]['acc']:.4f}", f"{report.get('1', {}).get('precision', 0):.4f}", f"{report.get('1', {}).get('recall', 0):.4f}", f"{report.get('1', {}).get('f1-score', 0):.4f}"])
    ax.table(cellText=table_data, colWidths=[0.2]*5, loc='center')
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_classification_report_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 2. Confusion Matrix Plot (Grid)
    fig, axs = plt.subplots(2, 4, figsize=(20, 10))
    axs = axs.ravel()
    for i, name in enumerate(names):
        sns.heatmap(metrics[name]['cm'], annot=True, fmt='d', cmap='Blues', ax=axs[i])
        axs[i].set_title(name)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_confusion_matrix_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 3. ROC-AUC Plot
    plt.figure(figsize=(10, 8))
    for name in names:
        plt.plot(metrics[name]['fpr'], metrics[name]['tpr'], label=f'{name} (AUC = {metrics[name]["auc"]:.2f})')
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('ROC-AUC Curves')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_roc_auc_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

# Plots if histories available
if histories:
    names_hist = list(histories.keys())
    
    # 4. Train-Val Accuracy vs Epochs
    plt.figure(figsize=(12, 8))
    for name in names_hist:
        plt.plot(histories[name]['accuracy'], label=f'{name} Train Acc')
        plt.plot(histories[name]['val_accuracy'], '--', label=f'{name} Val Acc')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Train-Val Accuracy vs Epochs')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_train_val_accuracy_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 5. Train-Val Loss vs Epochs
    plt.figure(figsize=(12, 8))
    for name in names_hist:
        plt.plot(histories[name]['loss'], label=f'{name} Train Loss')
        plt.plot(histories[name]['val_loss'], '--', label=f'{name} Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Train-Val Loss vs Epochs')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_train_val_loss_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

print("All plots generated.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization, LSTM, Conv1D, Conv2D,
                                     MaxPooling2D, Flatten, LeakyReLU, Input, Activation, SpatialDropout1D)

# Config
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

# Load test data
data = np.load(DATA_PATH, allow_pickle=True)
X_test = data['X_test']
y_test = data['y_test']

# Squeeze to 3D (samples, channels, timepoints)
X_test = np.squeeze(X_test, axis=1)  # Now (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels) for models expecting that format
X_test_3d = np.swapaxes(X_test, 1, 2)  # (samples, timepoints=125, channels=128)

# For CNN: Add channel dimension (samples, timepoints, channels, 1)
X_test_4d = np.expand_dims(X_test_3d, axis=-1)

# For flattened models: Flatten to 2D
X_test_flat = X_test_3d.reshape(X_test.shape[0], -1)

# Standardize X_test
scaler = StandardScaler()
X_test_flat_scaled = scaler.fit_transform(X_test_flat)

# Reshape back for 3D/4D
X_test_3d_scaled = X_test_flat_scaled.reshape(X_test_3d.shape)
X_test_4d_scaled = np.expand_dims(X_test_3d_scaled, axis=-1)

time_steps = X_test_3d.shape[1]  # 125
channels = X_test_3d.shape[2]    # 128

# Model metadata configuration
# Note: Stripped leading slashes from file names so os.path.join works correctly

model_names = ['TCN', 'GAN', 'FFNN', 'BNN with DVI', 'CNN', 'DNN', 'MLPNN', 'LSTM']
weight_list = [
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_tcn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/discriminator_dotprobe_main.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_ffnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_mc_dropout_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_cnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_dnn_model_dotprobe_main.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_mlpnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/best_lstm_model_dotprobe_main.weights.h5')
]
history_list = [
    os.path.join(OUTPUT_DIR, '/kaggle/working/tcn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/gan_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/ffnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/mc_dropout_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/cnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/dnn_training_history_dotprobe_enhanced_main.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/mlpnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, '/kaggle/working/lstm_training_history_dotprobe.npz')
]

# Build model_configs dynamically from lists
model_configs = []
for name, weights, history in zip(model_names, weight_list, history_list):
    if name == 'TCN':
        def build_tcn():
            inputs = Input(shape=(time_steps, channels))
            x = Conv1D(128, 3, padding='causal', dilation_rate=1)(inputs)
            x = BatchNormalization()(x)
            x = Activation('relu')(x)
            x = SpatialDropout1D(0.2)(x)
            x = Conv1D(128, 3, padding='causal', dilation_rate=2)(x)
            x = BatchNormalization()(x)
            x = Activation('relu')(x)
            x = SpatialDropout1D(0.2)(x)
            x = Conv1D(128, 3, padding='causal', dilation_rate=4)(x)
            x = BatchNormalization()(x)
            x = Activation('relu')(x)
            x = SpatialDropout1D(0.2)(x)
            x = Conv1D(128, 3, padding='causal', dilation_rate=8)(x)
            x = BatchNormalization()(x)
            x = Activation('relu')(x)
            x = SpatialDropout1D(0.2)(x)
            x = Flatten()(x)
            x = Dense(256, activation='relu')(x)
            x = Dropout(0.3)(x)
            outputs = Dense(1, activation='sigmoid')(x)
            return Model(inputs=inputs, outputs=outputs)
        build_func = build_tcn
        input_dim = (None, time_steps, channels)
    elif name == 'GAN':
        build_func = lambda: Sequential([
            Dense(512, input_shape=(time_steps * channels,)), 
            LeakyReLU(alpha=0.2), Dropout(0.4), 
            Dense(256), LeakyReLU(alpha=0.2), Dropout(0.4), 
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps * channels)
    elif name == 'FFNN':
        build_func = lambda: Sequential([
            Conv1D(64, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Flatten(),
            Dense(128), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(64), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(32), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    elif name == 'BNN with DVI(MC Dropout)':
        build_func = lambda: Sequential([
            Conv1D(64, kernel_size=3, padding='same', activation='relu', input_shape=(time_steps, channels)),  
            Dropout(0.5),
            Conv1D(32, kernel_size=3, padding='same', activation='relu'),
            Dropout(0.5),
            Flatten(),
            Dense(128, activation='relu'), Dropout(0.5),
            Dense(128, activation='relu'), Dropout(0.5),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    elif name == 'CNN':
        build_func = lambda: Sequential([
            Conv2D(32, (3, 3), activation='relu', input_shape=(time_steps, channels, 1)),
            BatchNormalization(), LeakyReLU(alpha=0.1), MaxPooling2D(pool_size=(2, 2)), Dropout(0.25),
            Conv2D(64, (3, 3), activation='relu'),
            BatchNormalization(), LeakyReLU(alpha=0.1), MaxPooling2D(pool_size=(2, 2)), Dropout(0.25),
            Flatten(),
            Dense(128, activation='relu'), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.5),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels, 1)
    elif name == 'DNN':
        build_func = lambda: Sequential([
            Conv1D(128, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(64, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Flatten(),
            Dense(512), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(256), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(128), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(64), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(32), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    elif name == 'MLPNN':
        build_func = lambda: Sequential([
            Conv1D(256, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(128, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(64, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Flatten(),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    elif name == 'LSTM':
        build_func = lambda: Sequential([
            LSTM(128, activation='tanh', return_sequences=True, input_shape=(time_steps, channels)),
            BatchNormalization(), Dropout(0.3),
            LSTM(64, activation='tanh', return_sequences=True),
            BatchNormalization(), Dropout(0.3),
            LSTM(32, activation='tanh'),
            BatchNormalization(), Dropout(0.3),
            Dense(64, activation='relu'), BatchNormalization(), Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    
    model_configs.append({
        'name': name,
        'weights': weights,
        'history': history,
        'build_func': build_func,
        'input_dim': input_dim
    })

metrics = {}
histories = {}

for cfg in model_configs:
    name = cfg['name']
    weights_path = cfg['weights']
    history_path = cfg['history']
    
    if not os.path.exists(weights_path):
        print(f"Skipping {name}: Weights not found at {weights_path}")
        continue
    
    # Instantiate architecture graph
    model = cfg['build_func']()
    
    # Explicitly build model shape before loading weights to initialize variables
    model.build(cfg['input_dim'])
    
    # Load model weights safely
    model.load_weights(weights_path)
    
    # Select input tensor according to architecture format
    if name == 'CNN':
        test_data = X_test_4d_scaled
    elif name == 'GAN':
        test_data = X_test_flat_scaled
    else:
        test_data = X_test_3d_scaled
    
    y_pred_prob = model.predict(test_data, verbose=0).ravel()
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    metrics[name] = {
        'acc': accuracy_score(y_test, y_pred),
        'report': classification_report(y_test, y_pred, output_dict=True),
        'cm': confusion_matrix(y_test, y_pred),
        'fpr': roc_curve(y_test, y_pred_prob)[0],
        'tpr': roc_curve(y_test, y_pred_prob)[1],
        'auc': roc_auc_score(y_test, y_pred_prob)
    }
    
    if os.path.exists(history_path):
        h = np.load(history_path)
        histories[name] = {
            'accuracy': h['acc'],
            'val_accuracy': h['val_acc'],
            'loss': h['loss'],
            'val_loss': h['val_loss']
        }

# Plots if metrics available
if metrics:
    names = list(metrics.keys())
    
    # 1. Classification Report Plot (Table)
    fig, ax = plt.subplots(figsize=(12, len(names) * 0.5 + 2))
    ax.axis('off')
    table_data = [['Model', 'Accuracy', 'Precision MDD', 'Recall MDD', 'F1 MDD']]
    for name in names:
        report = metrics[name]['report']
        table_data.append([
            name, 
            f"{metrics[name]['acc']:.4f}", 
            f"{report.get('1', {}).get('precision', 0):.4f}", 
            f"{report.get('1', {}).get('recall', 0):.4f}", 
            f"{report.get('1', {}).get('f1-score', 0):.4f}"
        ])
    ax.table(cellText=table_data, colWidths=[0.2]*5, loc='center')
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_classification_report_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 2. Confusion Matrix Plot (Grid)
    fig, axs = plt.subplots(2, 4, figsize=(20, 10))
    axs = axs.ravel()
    for i, name in enumerate(names):
        sns.heatmap(metrics[name]['cm'], annot=True, fmt='d', cmap='Blues', ax=axs[i])
        axs[i].set_title(name)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_confusion_matrix_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 3. ROC-AUC Plot
    plt.figure(figsize=(10, 8))
    for name in names:
        plt.plot(metrics[name]['fpr'], metrics[name]['tpr'], label=f'{name} (AUC = {metrics[name]["auc"]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('ROC-AUC Curves')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_roc_auc_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

# Plots if histories available
if histories:
    names_hist = list(histories.keys())
    
    # 4. Train-Val Accuracy vs Epochs
    plt.figure(figsize=(12, 8))
    for name in names_hist:
        plt.plot(histories[name]['accuracy'], label=f'{name} Train Acc')
        plt.plot(histories[name]['val_accuracy'], '--', label=f'{name} Val Acc')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Train-Val Accuracy vs Epochs')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_train_val_accuracy_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 5. Train-Val Loss vs Epochs
    plt.figure(figsize=(12, 8))
    for name in names_hist:
        plt.plot(histories[name]['loss'], label=f'{name} Train Loss')
        plt.plot(histories[name]['val_loss'], '--', label=f'{name} Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Train-Val Loss vs Epochs')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_train_val_loss_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

print("All plots generated successfully.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization, LSTM, Conv1D, Conv2D,
                                     MaxPooling2D, Flatten, LeakyReLU, Input, Activation, SpatialDropout1D)

# Config
OUTPUT_DIR = "/kaggle/working/dotprobe-erp-results"
DATA_PATH = os.path.join(OUTPUT_DIR, "dotprobe_data_split.npz")

# Load test data
data = np.load(DATA_PATH, allow_pickle=True)
X_test = data['X_test']
y_test = data['y_test']

# Squeeze to 3D (samples, channels, timepoints)
X_test = np.squeeze(X_test, axis=1)  # (samples, channels=128, timepoints=125)

# Swap axes to (samples, timepoints, channels)
X_test_3d = np.swapaxes(X_test, 1, 2)  # (samples, timepoints=125, channels=128)

# For CNN: Add channel dimension (samples, timepoints, channels, 1)
X_test_4d = np.expand_dims(X_test_3d, axis=-1)

# For flattened models: Flatten to 2D
X_test_flat = X_test_3d.reshape(X_test.shape[0], -1)

# Standardize X_test
scaler = StandardScaler()
X_test_flat_scaled = scaler.fit_transform(X_test_flat)

# Reshape back for 3D/4D
X_test_3d_scaled = X_test_flat_scaled.reshape(X_test_3d.shape)
X_test_4d_scaled = np.expand_dims(X_test_3d_scaled, axis=-1)

time_steps = X_test_3d.shape[1]  # 125
channels = X_test_3d.shape[2]    # 128

# Model metadata configuration
# Model name strings match the exact logic blocks below
model_names = ['TCN', 'GAN', 'FFNN', 'BNN with DVI(MC Dropout)', 'CNN', 'DNN', 'MLPNN', 'LSTM']

# Correct path joins without leading slashes inside join calls
weight_list = [
    os.path.join(OUTPUT_DIR, 'best_tcn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, 'discriminator_dotprobe_main.weights.h5'),
    os.path.join(OUTPUT_DIR, 'best_ffnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, 'best_mc_dropout_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, 'best_cnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, 'best_dnn_model_dotprobe_main.weights.h5'),
    os.path.join(OUTPUT_DIR, 'best_mlpnn_model_dotprobe.weights.h5'),
    os.path.join(OUTPUT_DIR, 'best_lstm_model_dotprobe_main.weights.h5')
]
history_list = [
    os.path.join(OUTPUT_DIR, 'tcn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, 'gan_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, 'ffnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, 'mc_dropout_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, 'cnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, 'dnn_training_history_dotprobe_enhanced_main.npz'),
    os.path.join(OUTPUT_DIR, 'mlpnn_training_history_dotprobe.npz'),
    os.path.join(OUTPUT_DIR, 'lstm_training_history_dotprobe.npz')
]

# Build model_configs dynamically from lists
model_configs = []
for name, weights, history in zip(model_names, weight_list, history_list):
    if name == 'TCN':
        def build_tcn():
            inputs = Input(shape=(time_steps, channels))
            x = Conv1D(128, 3, padding='causal', dilation_rate=1)(inputs)
            x = BatchNormalization()(x)
            x = Activation('relu')(x)
            x = SpatialDropout1D(0.2)(x)
            x = Conv1D(128, 3, padding='causal', dilation_rate=2)(x)
            x = BatchNormalization()(x)
            x = Activation('relu')(x)
            x = SpatialDropout1D(0.2)(x)
            x = Conv1D(128, 3, padding='causal', dilation_rate=4)(x)
            x = BatchNormalization()(x)
            x = Activation('relu')(x)
            x = SpatialDropout1D(0.2)(x)
            x = Conv1D(128, 3, padding='causal', dilation_rate=8)(x)
            x = BatchNormalization()(x)
            x = Activation('relu')(x)
            x = SpatialDropout1D(0.2)(x)
            x = Flatten()(x)
            x = Dense(256, activation='relu')(x)
            x = Dropout(0.3)(x)
            outputs = Dense(1, activation='sigmoid')(x)
            return Model(inputs=inputs, outputs=outputs)
        build_func = build_tcn
        input_dim = (None, time_steps, channels)
    elif name == 'GAN':
        build_func = lambda: Sequential([
            Dense(512, input_shape=(time_steps * channels,)), 
            LeakyReLU(alpha=0.2), Dropout(0.4), 
            Dense(256), LeakyReLU(alpha=0.2), Dropout(0.4), 
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps * channels)
    elif name == 'FFNN':
        build_func = lambda: Sequential([
            Conv1D(64, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Flatten(),
            Dense(128), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(64), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(32), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    elif name == 'BNN with DVI(MC Dropout)':
        build_func = lambda: Sequential([
            Conv1D(64, kernel_size=3, padding='same', activation='relu', input_shape=(time_steps, channels)),  
            Dropout(0.5),
            Conv1D(32, kernel_size=3, padding='same', activation='relu'),
            Dropout(0.5),
            Flatten(),
            Dense(128, activation='relu'), Dropout(0.5),
            Dense(128, activation='relu'), Dropout(0.5),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    elif name == 'CNN':
        build_func = lambda: Sequential([
            Conv2D(32, (3, 3), activation='relu', input_shape=(time_steps, channels, 1)),
            BatchNormalization(), LeakyReLU(alpha=0.1), MaxPooling2D(pool_size=(2, 2)), Dropout(0.25),
            Conv2D(64, (3, 3), activation='relu'),
            BatchNormalization(), LeakyReLU(alpha=0.1), MaxPooling2D(pool_size=(2, 2)), Dropout(0.25),
            Flatten(),
            Dense(128, activation='relu'), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.5),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels, 1)
    elif name == 'DNN':
        build_func = lambda: Sequential([
            Conv1D(128, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(64, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Flatten(),
            Dense(512), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(256), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(128), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(64), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(32), BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    elif name == 'MLPNN':
        build_func = lambda: Sequential([
            Conv1D(256, kernel_size=3, padding='same', input_shape=(time_steps, channels)),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(128, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(64, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Conv1D(32, kernel_size=3, padding='same'),
            BatchNormalization(), LeakyReLU(alpha=0.1), Dropout(0.3),
            Flatten(),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    elif name == 'LSTM':
        build_func = lambda: Sequential([
            LSTM(128, activation='tanh', return_sequences=True, input_shape=(time_steps, channels)),
            BatchNormalization(), Dropout(0.3),
            LSTM(64, activation='tanh', return_sequences=True),
            BatchNormalization(), Dropout(0.3),
            LSTM(32, activation='tanh'),
            BatchNormalization(), Dropout(0.3),
            Dense(64, activation='relu'), BatchNormalization(), Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])
        input_dim = (None, time_steps, channels)
    
    model_configs.append({
        'name': name,
        'weights': weights,
        'history': history,
        'build_func': build_func,
        'input_dim': input_dim
    })

metrics = {}
histories = {}

for cfg in model_configs:
    name = cfg['name']
    weights_path = cfg['weights']
    history_path = cfg['history']
    
    # Fallback path checking if files reside directly in /kaggle/working/
    if not os.path.exists(weights_path):
        fallback_path = os.path.join("/kaggle/working", os.path.basename(weights_path))
        if os.path.exists(fallback_path):
            weights_path = fallback_path
        else:
            print(f"Skipping {name}: Weights not found at {weights_path}")
            continue

    # Instantiate architecture graph
    model = cfg['build_func']()
    
    # Explicitly build model shape before loading weights
    model.build(cfg['input_dim'])
    
    # Load model weights safely
    model.load_weights(weights_path)
    
    # Select input tensor according to architecture format
    if name == 'CNN':
        test_data = X_test_4d_scaled
    elif name == 'GAN':
        test_data = X_test_flat_scaled
    else:
        test_data = X_test_3d_scaled
    
    # Predict step (Handling MC Dropout Monte Carlo Inference vs standard Deterministic pass)
    if name == 'BNN with DVI(MC Dropout)':
        T = 50
        mc_predictions = []
        X_test_tensor = tf.convert_to_tensor(test_data, dtype=tf.float32)
        for _ in range(T):
            preds = model(X_test_tensor, training=True)
            mc_predictions.append(preds.numpy())
        
        mc_predictions = np.array(mc_predictions)
        y_pred_prob = np.mean(mc_predictions, axis=0).ravel()
        y_pred_variance = np.var(mc_predictions, axis=0).ravel()
        print(f"[{name}] Mean Uncertainty (Predictive Variance): {np.mean(y_pred_variance):.4f}")
    else:
        y_pred_prob = model.predict(test_data, verbose=0).ravel()
        
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    metrics[name] = {
        'acc': accuracy_score(y_test, y_pred),
        'report': classification_report(y_test, y_pred, output_dict=True),
        'cm': confusion_matrix(y_test, y_pred),
        'fpr': roc_curve(y_test, y_pred_prob)[0],
        'tpr': roc_curve(y_test, y_pred_prob)[1],
        'auc': roc_auc_score(y_test, y_pred_prob)
    }
    
    if not os.path.exists(history_path):
        fallback_hist_path = os.path.join("/kaggle/working", os.path.basename(history_path))
        if os.path.exists(fallback_hist_path):
            history_path = fallback_hist_path

    if os.path.exists(history_path):
        h = np.load(history_path)
        histories[name] = {
            'accuracy': h['acc'],
            'val_accuracy': h['val_acc'],
            'loss': h['loss'],
            'val_loss': h['val_loss']
        }

# Plots if metrics available
if metrics:
    names = list(metrics.keys())
    
    # 1. Classification Report Plot (Table)
    fig, ax = plt.subplots(figsize=(12, len(names) * 0.5 + 2))
    ax.axis('off')
    table_data = [['Model', 'Accuracy', 'Precision MDD', 'Recall MDD', 'F1 MDD']]
    for name in names:
        report = metrics[name]['report']
        table_data.append([
            name, 
            f"{metrics[name]['acc']:.4f}", 
            f"{report.get('1', {}).get('precision', 0):.4f}", 
            f"{report.get('1', {}).get('recall', 0):.4f}", 
            f"{report.get('1', {}).get('f1-score', 0):.4f}"
        ])
    ax.table(cellText=table_data, colWidths=[0.2]*5, loc='center')
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_classification_report_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 2. Confusion Matrix Plot (Grid)
    fig, axs = plt.subplots(2, 4, figsize=(20, 10))
    axs = axs.ravel()
    for i, name in enumerate(names):
        sns.heatmap(metrics[name]['cm'], annot=True, fmt='d', cmap='Blues', ax=axs[i])
        axs[i].set_title(name)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_confusion_matrix_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 3. ROC-AUC Plot
    plt.figure(figsize=(10, 8))
    for name in names:
        plt.plot(metrics[name]['fpr'], metrics[name]['tpr'], label=f'{name} (AUC = {metrics[name]["auc"]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('ROC-AUC Curves')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_roc_auc_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

# Plots if histories available
if histories:
    names_hist = list(histories.keys())
    
    # 4. Train-Val Accuracy vs Epochs
    plt.figure(figsize=(12, 8))
    for name in names_hist:
        plt.plot(histories[name]['accuracy'], label=f'{name} Train Acc')
        plt.plot(histories[name]['val_accuracy'], '--', label=f'{name} Val Acc')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Train-Val Accuracy vs Epochs')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_train_val_accuracy_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

    # 5. Train-Val Loss vs Epochs
    plt.figure(figsize=(12, 8))
    for name in names_hist:
        plt.plot(histories[name]['loss'], label=f'{name} Train Loss')
        plt.plot(histories[name]['val_loss'], '--', label=f'{name} Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Train-Val Loss vs Epochs')
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dotprobe_all_train_val_loss_main_best_final_last.png'), bbox_inches='tight')
    plt.show()
    plt.close()

print("All plots generated successfully.")